In [ ]:


import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 1561312


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)
generator = torch.Generator().manual_seed(SEED)

In [ ]:
torch.random.initial_seed()

In [ ]:
model_path = "google/siglip-base-patch16-224"
device = "cuda"

In [ ]:
from transformers import SiglipProcessor

processor = SiglipProcessor.from_pretrained(model_path)

In [ ]:

from transformers import SiglipProcessor, SiglipModel, SiglipTokenizer, SiglipTextConfig
import torch

def load_base_model():

    trained_model = SiglipModel.from_pretrained(
        model_path,
        device_map=device,
        # torch_dtype=torch.bfloat16
    )


    trained_model = trained_model.to(torch.bfloat16)
    trained_model = torch.compile(trained_model) 

    return trained_model

In [ ]:
train_dataset = "/run/media/victor/pessoal/mestrado/codigo/datasets/V4_train_art.csv"
test_dataset = "/run/media/victor/pessoal/mestrado/codigo/datasets/V4_test_art.csv"

In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:
from seq_aux_dataloader_aug import imageTextDataset
from torch.utils.data import ConcatDataset

train_data = ConcatDataset([
    imageTextDataset(train_dataset, processor)
])

test_data = ConcatDataset([ 
    imageTextDataset(test_dataset, processor),
    ])

In [ ]:
train_data.datasets[0].image_paths

In [ ]:
configs = [

    {
        "lr" : 5e-5,
        "batch_size" : 100
    },
    {
        "lr" : 5e-5,
        "batch_size" : 200
    },
    {
        "lr" : 5e-5,
        "batch_size" : 150
    },
    {
        "lr" : 2.5e-5,
        "batch_size" : 200
    },
    {
        "lr" : 2.5e-5,
        "batch_size" : 150
    },
    {
        "lr" : 2.5e-5,
        "batch_size" : 100
    }
]

In [ ]:
from seq_aux_dataloader_aug import train
import gc

modelo_base= "phase_1"

for config in configs:
    print(config)

    trained_model = load_base_model()

    trained_model = train(train_data,test_data, trained_model, processor, lr=config["lr"], batch_size=config["batch_size"], num_workers = 2, seed=SEED, modelo_base=modelo_base, only_projection=False, folder="V4_phase_1")

    del trained_model
    
    gc.collect()
    torch.cuda.empty_cache()
